# 3551. 数位和排序需要的最小交换次数

这份笔记给出两个完整解法：Solution 1 用标准 BFS 模板遍历位置关系，Solution 2 用并查集分组。两种方法都先确定每个数字的目标位置，再统计位置关系形成的圈。

## 先看懂两种方法共同做的事

数位和就是把数字的每一位加起来，例如 `20` 的数位和是 `2 + 0 = 2`。先按数位和从小到大排；数位和相同时，再按数字本身从小到大排。题目允许交换任意两个位置，不要求相邻。

用 `[20, 3, 1]` 举例，目标顺序是 `[1, 20, 3]`。下标就是位置编号，从 `0` 开始：

| 现在的位置 i | 现在放着什么数字 | 这个数字应该去哪里 to[i] |
| --- | --- | --- |
| 0 | 20 | 1 |
| 1 | 3 | 2 |
| 2 | 1 | 0 |

所以 `to = [1, 2, 0]`。`to[i]` 的意思是“位置 i 上的数字应该去的位置”，不是目标位置上放的数字。

沿着“应该去哪里”一直走：`0 → 1 → 2 → 0`，最后回到了起点。我们把这种首尾相接的一圈叫作**环**。所有数字都不同，所以每个位置都有唯一的下一站，也只有一个位置指向它；全部位置会分成若干个互不重叠的圈。已经归位的位置自己指向自己，也算一圈。

## 一圈为什么需要“位置个数减一”次交换？

先实际交换看看：

```text
20  3  1
↓ 交换位置 0 和位置 2，把 1 放回正确位置
 1  3 20
↓ 交换位置 1 和位置 2，把 20 放回正确位置
 1 20  3
```

三个位置两次就完成了。最后那个数字会跟着归位，不需要单独处理。一般来说，一圈有 `k` 个位置，就能用 `k - 1` 次交换归位：每次拿出一个数字放回正确位置，最后一个自动归位。一个位置的小圈已经正确，需要零次交换。

为什么不能更少？一次交换最多把一个圈拆成两个圈，也就是圈数最多增加一。最终每个位置都自己一圈，共有 `n` 圈；起初有 `c` 圈，至少要交换 `n - c` 次。分别处理每一圈恰好能达到这个次数。

因此，可以把每一圈的 `k - 1` 加起来，也可以计算 `总位置数 - 圈数`。两个解法只是找出这些圈的方式不同。初次阅读时先记住三个位子两次交换的例子，再读代码。

## Solution 1. BFS：用标准队列模板清点一圈的位置

BFS 可以理解成排队办事：先来先处理。`deque` 是队列，`append` 从队尾加入，`popleft` 从队头取出。

### 先认识邻接表 graph

“邻居”指从当前位置能直接走到的位置。`graph[current]` 存放当前位置的所有邻居，`for neighbor in graph[current]` 就是逐个检查它们。这种记录方式叫邻接表。

本题每个位置只有一个下一站。对于共同例子的 `to = [1, 2, 0]`，邻接表是 `graph = [[1], [2], [0]]`：位置 0 的邻居列表是 `[1]`，位置 1 的是 `[2]`，位置 2 的是 `[0]`。虽然每个列表只有一个位置，仍然统一使用遍历邻居的标准模板。

### 标准 BFS 模板的五个步骤

1. 创建队列，让起点入队，同时在 `visited` 上签到。
2. 只要队列不空，就从队头取出一个位置。
3. 遍历这个位置的所有邻居。
4. 跳过已经签到的邻居；没签到的先签到，再放进队尾。
5. 重复上述过程，直到队列为空。

代码把这五步封装成 `bfs(start)`。`count` 只是本题增加的统计：每处理一个位置就加一，最后返回这一圈有多少个位置。外层循环负责从尚未签到的位置开始下一圈，并累加每圈的 `位置数 - 1`。

这里 BFS 遍历位置关系，不枚举交换后可能出现的数组。无需按层统计，因为访问层数不是交换次数；交换次数来自前面证明的环性质。标准 BFS 可以只遍历节点，求最短距离或按层处理时才需要额外记录距离或分层。

In [14]:
from collections import deque
from typing import List


class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        def calc_dig_sum(num: int) -> int:
            return sum(int(digit) for digit in str(num))

        ordered_nums = sorted(nums, key=lambda num: (calc_dig_sum(num), num))
        target_index = {num: i for i, num in enumerate(ordered_nums)}
        # graph[i] 存放位置 i 能直接走到的邻居，本题只有一个。
        graph = [[target_index[num]] for num in nums]
        print(graph)

        n = len(nums)
        visited = [False] * n

        def bfs(start: int) -> int:
            queue = deque([start])
            visited[start] = True  # 入队时签到，避免重复入队。
            count = 0

            while queue:
                current = queue.popleft()
                count += 1

                for neighbor in graph[current]:
                    if visited[neighbor]:
                        continue
                    visited[neighbor] = True
                    queue.append(neighbor)

            return count

        ans = 0
        for start in range(n):
            if not visited[start]:
                cycle_size = bfs(start)
                ans += cycle_size - 1

        return ans


# 保留 BFS 类，方便与后面的并查集解法一起验证。
SolutionBFS = Solution

### 写完代码，再用大白话走一遍 BFS

对于 `to = [1, 2, 0]`，队列变化如下。每个位置在排队时就签到，防止重复排队。

| 操作 | 操作后的队列 | 已经处理的位置数 |
| --- | --- | --- |
| 让起点 0 签到并排队 | `[0]` | 0 |
| 取出 0；让下一站 1 签到并排队 | `[1]` | 1 |
| 取出 1；让下一站 2 签到并排队 | `[2]` | 2 |
| 取出 2；下一站 0 已签到，不再排队 | `[]` | 3 |

队列空了，说明这一圈已经清点完。它有三个位置，答案加两次交换。还有没签到的位置，就从那里再开一轮队列。

`for neighbor in graph[current]` 在这个例子里每次只检查一个邻居。比如处理位置 2 时，邻居是 0；0 已签到，就跳过。没有新的位置入队，队列自然变空。

`visited` 不需要改回 `False`：数过的位置以后不应该再数。代码也没有真的交换数字，只是计算每圈的最少交换次数。

数位和计算与排序用时 `O(nd + n log n)`，BFS 用时 `O(n)`，额外空间 `O(n)`；本题总时间可记作 `O(n log n)`。

## Solution 2. 并查集（Union-Find）

先用一句大白话理解：**并查集就是一本分组登记册。`union(a, b)` 把两边所在的小组合起来，`find(x)` 查这个位置所在组的代表是谁。代表相同，就属于同一组。**

这里登记的是位置编号。对于 `0 → 1 → 2 → 0`，先把 `0` 和 `1` 分成一组，再把 `1` 和 `2` 分成一组，于是三个位置就是同一组。最后处理 `2` 和 `0` 时，两边已经同组，不需要重复合并。

BFS 会把一圈走一遍并数位置；并查集把有关联的位置合成一组，最后统计还剩几组。此处的一组正好对应一圈。


### 1. 适用问题与常见题型

**核心用途：维护一批元素的分组，反复合并集合，并查询两个元素是否属于同一组。** 合并具有传递性：若 A 与 B 同组、B 与 C 同组，那么 A 与 C 也同组。普通并查集尤其适合关系不断加入、集合只合并不拆分的场景。

| 常见题型 | 如何建模与套用模板 |
| --- | --- |
| 无向图连通性、连通分量数量或大小 | 节点作为元素；对每条边执行 `union(u, v)`，用根判断连通，用 `components` 或根上的 `size` 统计。 |
| 动态加边、在线连通性查询 | 每加一条边就合并两端；随后可随时查询两个节点是否连通，不必重新遍历整张图。 |
| 账号合并、朋友圈、相同属性归组 | 找到明确属于同组的对象并合并；字符串等对象可以先映射为整数编号。合并后通常还需按根收集成员。 |
| 相等与不等约束是否矛盾 | 先合并所有相等关系，再检查每个不等关系的两端是否已同组；同组就有矛盾。不能把不等关系直接当作合并操作。 |
| 无向图判环、寻找冗余边 | 逐条处理边；若两端已有相同根，加入该边会形成环，对应 `union(u, v)` 返回 `False`。 |
| Kruskal 最小生成树 | 边按权重从小到大处理；仅在 `union(u, v)` 成功时选入该边并累计权重。图不连通时只能得到最小生成森林。 |
| 网格连通块、动态增加陆地 | 将坐标映射为 `r * cols + c`，合并相邻的有效格子。动态加点时另行维护激活标记和有效分量数，不能把水域计入岛屿。 |
| 交换限制、排列归位 | 若允许沿指定边交换，连通分量内的元素可以重新排列；本题则合并当前位置与目标位置，通过置换环的性质计算最少交换次数。 |

**读题时的识别信号**

1. 是否反复出现“连接、合并、同一组、能否互通、集合个数、集合大小”等操作或查询？
2. 能否把对象编号，把关系表示为“这两个对象应归入同一集合”？
3. 是否只需分组结果，而无需具体路径？若还存在频繁加边，并查集通常很合适。

**边界与选型**

- 对一张静态图只统计一次连通块，BFS 也很直接；并查集在多次合并和查询时更有优势。
- 普通并查集不记录实际路径或距离，不能直接求最短路；无权最短路通常考虑 BFS，非负权最短路可考虑 Dijkstra。`parent` 中的边是集合维护关系，不一定是原图的边。
- 普通并查集忽略方向，不能直接判断有向可达性或有向环。本题能使用它，是因为已经证明位置映射形成互不相交的环。
- 普通并查集不支持直接删除边或拆分集合；部分此类题可离线倒序转为加边，或使用回滚并查集等扩展。
- 比值、差值、奇偶性等关系需要额外维护信息，通常使用带权并查集，不能直接套用下面的基础模板。
- 本题的 `n - components` 来自置换环的最少交换性质，不是所有交换问题或连通分量问题的通用答案。

### 2. 通用模板

并查集维护元素所属的集合，适合处理连通性和集合合并。这里使用从 `0` 到 `n - 1` 的编号。

- `parent[x]`：节点的父节点；根节点满足 `parent[x] == x`。
- `size[root]`：以该根为代表的集合大小，仅根节点上的值有效。
- `components`：当前集合数，初始化为 `n`；每次成功合并减一。
- `find(x)`：查找根，并通过路径压缩加速后续查询。
- `union(a, b)`：按大小合并两个集合；已连通时返回 `False`，成功合并时返回 `True`。

判断两个节点是否连通：`uf.find(a) == uf.find(b)`。模板使用迭代实现，避免递归深度问题。

In [15]:
class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            # 路径折半：让当前节点直接指向祖父节点。
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        root_a, root_b = self.find(a), self.find(b)
        if root_a == root_b:
            return False

        # 小集合挂到大集合的根节点下。
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a
        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        self.components -= 1
        return True


### 3. 套用到 3551

先按 `(数位和, 数值)` 排序，得到目标数组，再用字典记录每个值的目标下标。对原数组中的每个位置 `i`，合并 `i` 与 `target_index[nums[i]]`。并查集的节点是**下标**，不是数值或数位和。

元素互不相同，所以每个位置恰好指向一个目标位置，也恰好被一个位置指向。这些关系形成若干互不相交的置换环；忽略方向后，每个环恰好是一个连通分量。并查集用来统计这些分量，不需要存储环的访问顺序。

**为什么答案是 `n - uf.components`？**

长度为 `k` 的环，可以每次把一个元素交换到目标位置，用 `k - 1` 次交换完成归位，最后一个元素会自动归位。另一方面，一次交换最多让置换的环数增加一；目标排列有 `n` 个长度为一的环，因此从 `c` 个环出发至少需要 `n - c` 次交换。分别处理各环恰好达到这个下界。已归位的位置是长度为一的环，贡献零次交换。

例如 `[18, 43, 34, 16]` 的目标数组为 `[16, 34, 43, 18]`，对应关系为 `0 → 3 → 0` 和 `1 → 2 → 1`。合并后有两个分量，因此需要 `4 - 2 = 2` 次交换。

注意：`union` 修改的是集合关系，没有实际交换数组元素。成功合并的总次数也等于 `n - uf.components`。

设最大数位数为 `d`：计算数位和需要 `O(nd)`，排序需要 `O(n log n)`，并查集操作总计摊还 `O(n α(n))`；空间 `O(n)`。本题 `d ≤ 10`，总时间可记为 `O(n log n)`。

In [16]:
from typing import List


class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        def digit_sum(num: int) -> int:
            total = 0
            while num:
                total += num % 10
                num //= 10
            return total

        ordered_nums = sorted(nums, key=lambda num: (digit_sum(num), num))
        target_index = {num: i for i, num in enumerate(ordered_nums)}

        uf = UnionFind(len(nums))
        for i, num in enumerate(nums):
            uf.union(i, target_index[num])

        return len(nums) - uf.components


# 在力扣提交时，需同时复制上面的 UnionFind 类。


SolutionDSU = Solution


### 写完代码，再用大白话走一遍并查集

仍然看 `[20, 3, 1]`，我们给三个位置分组：

```text
开始：每个位置自己一组      {0} {1} {2}   共 3 组
20 在位置 0，要去位置 1    {0, 1} {2}    共 2 组
 3 在位置 1，要去位置 2    {0, 1, 2}     共 1 组
 1 在位置 2，要去位置 0    {0, 1, 2}     还是 1 组
```

代码里的 `uf.union(i, target_index[num])`，就是“把现在的位置和应该去的位置归为同组”。`uf.components` 是组数，所以最后返回 `3 - 1 = 2`。

如果分成了两组，一组有 3 个位置，另一组有 2 个位置，分别需要 `3 - 1 = 2` 次、`2 - 1 = 1` 次交换，合计 `3` 次。也就是 `总位置数 5 - 组数 2 = 3`。这说明并查集只需知道组数，不必逐个计算每组的交换次数。

**模板里暂时不熟悉的变量，可以这样读：**

- `parent` 是“找组代表时，下一步该找谁”的登记表；顺着它找，最终会找到自己指向自己的代表。它不保存数字的目标位置，别和 `to` 混淆。
- `size` 记录组有多大；合并时让小组跟随大组，是为了以后找代表更快。
- 路径压缩相当于记住找代表的捷径，下次少找几个人。它只让查找更快，不改变谁和谁同组。
- `components` 记录还剩几组，只有真的把两个不同组并成一个时才减一。

先理解“查组”和“合组”，再回头学习这些加速细节即可。并查集也没有真的交换原数组。

## 两种方法对照与运行方式

| 方法 | 大白话 | 代码怎么得到答案 |
| --- | --- | --- |
| BFS | 让待处理的位置排队，依次清点 | 每清点完一圈，加上“这一圈位置数减一” |
| 并查集 | 把有关联的位置登记到同一组 | 总位置数减去最后剩下的组数 |

学习时先看共同例子，再看 BFS 的队列和邻接表，最后看并查集的分组。两个方法都不会修改输入数组，总时间 `O(n log n)`，额外空间 `O(n)`。

在 Jupyter 中从上到下运行即可。`SolutionBFS` 和 `SolutionDSU` 是为了保留两个类，避免后面定义的 `Solution` 覆盖前面的名字。

提交到力扣时：BFS 复制 Solution 1 的代码单元格即可，末尾别名可以省略；并查集需要同时复制 `UnionFind` 模板和 Solution 2 的 `Solution` 类。

In [17]:
cases = [
    ([37, 100], 1),
    ([22, 14, 33, 7], 0),
    ([18, 43, 34, 16], 2),
    ([1], 0),
    ([40, 4, 13], 2),  # 数位和相同，按数字本身排序。
    ([20, 3, 1], 2),   # 上面逐步解释的三位置环。
]

for name, solver_class in [
    ('Solution 1: BFS', SolutionBFS),
    ('Solution 2: Union-Find', SolutionDSU),
]:
    solver = solver_class()
    for nums, expected in cases:
        actual = solver.minSwaps(nums)
        assert actual == expected, (name, nums, actual, expected)
    print(f'{name}: {len(cases)} examples passed; [20, 3, 1] -> {solver.minSwaps([20, 3, 1])}')

[[1], [0]]
[[0], [1], [2], [3]]
[[3], [2], [1], [0]]
[[0]]
[[2], [0], [1]]
[[1], [2], [0]]
[[1], [2], [0]]
Solution 1: BFS: 6 examples passed; [20, 3, 1] -> 2
Solution 2: Union-Find: 6 examples passed; [20, 3, 1] -> 2
